In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = "../data/raw"

In [ ]:
base_cadastral = pd.read_parquet(
    f"{DATA_PATH}/base_cadastral.parquet"
)

base_submissao = pd.read_parquet(
    f"{DATA_PATH}/base_submissao.parquet"
)

historico_emprestimos = pd.read_parquet(
    f"{DATA_PATH}/historico_emprestimos.parquet"
)

historico_parcelas = pd.read_parquet(
    f"{DATA_PATH}/historico_parcelas.parquet"
)

In [ ]:
bases = {
    "base_cadastral": base_cadastral,
    "base_submissao": base_submissao,
    "historico_emprestimos": historico_emprestimos,
    "historico_parcelas": historico_parcelas
}

for nome, df in bases.items():

    print("="*80)
    print(nome)
    print("="*80)

    print(f"Shape: {df.shape}")

    display(
        pd.DataFrame({
            "dtype": df.dtypes,
            "missing": df.isna().sum(),
            "missing_pct": (
                df.isna().mean()*100
            ).round(2)
        })
    )

In [ ]:
clientes_emprestimos = set(
    historico_emprestimos["id_cliente"]
)

clientes_cadastral = set(
    base_cadastral["id_cliente"]
)

clientes_sem_cadastro = (
    clientes_emprestimos -
    clientes_cadastral
)

print(
    f"Clientes sem cadastro: "
    f"{len(clientes_sem_cadastro)}"
)

In [ ]:
contratos_emprestimos = set(
    historico_emprestimos["id_contrato"]
)

contratos_parcelas = set(
    historico_parcelas["id_contrato"]
)

contratos_sem_parcelas = (
    contratos_emprestimos -
    contratos_parcelas
)

print(
    f"Contratos sem parcelas: "
    f"{len(contratos_sem_parcelas)}"
)

In [ ]:
cols_data_emprestimo = [
    "data_decisao",
    "data_liberacao",
    "data_primeiro_vencimento",
    "data_ultimo_vencimento_original",
    "data_ultimo_vencimento",
    "data_encerramento"
]

for col in cols_data_emprestimo:
    historico_emprestimos[col] = pd.to_datetime(
        historico_emprestimos[col]
    )

parcela_datas = [
    "data_prevista_pagamento",
    "data_real_pagamento"
]

for col in parcela_datas:
    historico_parcelas[col] = pd.to_datetime(
        historico_parcelas[col]
    )

base_submissao["data_solicitacao"] = pd.to_datetime(
    base_submissao["data_solicitacao"]
)

base_cadastral["data_nascimento"] = pd.to_datetime(
    base_cadastral["data_nascimento"]
)

In [ ]:
print(
    historico_emprestimos[
        "data_decisao"
    ].agg(
        ["min","max"]
    )
)

print(
    historico_parcelas[
        "data_prevista_pagamento"
    ].agg(
        ["min","max"]
    )
)

In [ ]:
contratos_cliente = (
    historico_emprestimos
    .groupby("id_cliente")
    ["id_contrato"]
    .nunique()
)

contratos_cliente.describe()

In [ ]:
population_train = (
    historico_emprestimos
    .query(
        "status_contrato != 'Recusado'"
    )
    .copy()
)

In [ ]:
population_score = (
    base_submissao
    .copy()
)

In [ ]:
output_path = (
    "../data/processed"
)

Path(output_path).mkdir(
    parents=True,
    exist_ok=True
)

population_train.to_parquet(
    f"{output_path}/population_train.parquet",
    index=False
)

population_score.to_parquet(
    f"{output_path}/population_score.parquet",
    index=False
)